## 1. RNN（Recurrent Neural Network）

`RNN`（再帰型ニューラルネットワーク）は、**順序を持つデータを学習する深層学習モデル**である。

時系列データやテキストなどの<strong>順序データ（Sequential Data）</strong>から特徴を学習し、  
**分類・回帰の両方**に利用できる。

- **特徴**：前の時点の情報を、次の時点の学習に利用する。
- **主な用途**：株価予測、時系列分析、自然言語処理など
- **基本構造**：現在の入力と前の時点の隠れ状態を使って、新しい隠れ状態を計算する。
- **一般的なニューラルネットワークとの違い**：RNNは前の時点の情報を次の時点へ伝える。

<br>

#### ※ RNNにおける回帰・分類

<table style="
width:950px;
max-width:100%;
border-collapse:collapse;
color:#000;
text-align:left;
margin-left:0 !important;
margin-right:auto !important;
">
<tbody>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:10px 18px; width:180px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>回帰</strong></span><br>
<strong>（Regression）</strong>
</td>

<td style="padding:10px 18px; color:#000; line-height:1.8;">
<strong>目的：</strong>連続した数値を予測する。<br>
<span style="color:#ff4d5a;">例：</span>翌日の株価、次週の気温、将来の電力使用量<br>
<strong>出力層：</strong>活性化関数なし（または Linear）のニューロンを使い、連続値を出力する。<br>
<strong>損失関数：</strong><span style="color:#ff8c00;"><strong>MSE・MAE</strong></span> などを使用する。
</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:10px 18px; width:180px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>分類</strong></span><br>
<strong>（Classification）</strong>
</td>

<td style="padding:10px 18px; color:#000; line-height:1.8;">
<strong>目的：</strong>カテゴリやラベルを予測する。<br>
<span style="color:#ff4d5a;">例：</span>感情分析、スパムメール判定、音声コマンド認識<br>
<strong>出力層：</strong><span style="color:#ff8c00;"><strong>Softmax・Sigmoid</strong></span> を使ってクラスの確率を出力する。<br>
<strong>損失関数：</strong><span style="color:#ff8c00;"><strong>Cross-Entropy</strong></span> を使用する。
</td>
</tr>

</tbody>
</table>

<br>

#### ■ RNN予測の考え方

過去の一定期間のデータを使って、**次の時点の値を予測する**。

<br>

例)

**1～3日目の株価 → RNNで学習 → 4日目の株価を予測**

- 入力区間を1日ずつ移動させる**スライディングウィンドウ**を使って、複数の学習サンプルを作成する。

<br>

In [ ]:
import numpy as np

X = np.array([1, 2, 3, 4, 5, 6])

display(X.shape)             # (6,)

display(X.reshape(2, 3))     # 2行 × 3列

display(X.reshape(3, 2))     # 3行 × 2列

display(X.reshape(3, 2, 1))  # 3 × 2 × 1 の3次元配列

(6,)

array([[1, 2, 3],
       [4, 5, 6]])

array([[1, 2],
       [3, 4],
       [5, 6]])

array([[[1],
        [2]],

       [[3],
        [4]],

       [[5],
        [6]]])

<Br>

<br>

#### ■ スライディングウィンドウ

過去3日分を**入力データ**、その次の日を**正解データ**として学習サンプルを作る。

- `price[i:i+3]`：過去3日分の入力データ
- `price[i+3]`：次の日の正解データ
- `price[-3:]`：全データの最後の3日分

<span style="color:red;">
入力：過去3日 → 正解：次の1日
</span>

<br>

## 【RNNによる株価予測】

仮想の株価データを使って、**過去3日分の株価から次の日の株価を予測するRNN回帰モデル**を構築・学習する。

<br>

#### ① 株価データ準備

RNNの回帰を練習するために、仮想の株価データを用意する。

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense

price = np.array([
    100, 102, 101, 105, 107, 106, 110, 112, 115, 113,
    118, 120, 119, 123, 125, 128, 126, 130, 133, 135
], dtype=float)

<Br>

#### ② 正規化

入力値を小さな範囲に変換し、ニューラルネットワークの学習を安定させる。

<br>

##### ※ 主な正規化方法

- **一定の値で割る**：今回のように `150` などで割って、値の大きさを小さくする。
- **MaxAbsScaler**：各値を**最大絶対値**で割り、通常 `-1～1` の範囲に変換する。
- **MinMaxScaler**：最小値を `0`、最大値を `1` として、**0～1の範囲**に変換する。

<span style="color:red;">
今回の <strong>/ 150</strong> は仕組みを理解するための簡単な方法であり、実際の前処理では <strong>Min-Max</strong> などのスケーリング方法も使用される。
</span>

In [3]:
price = price / 150

<br>

#### ③ 学習データ作成

**スライディングウィンドウ**を使って、過去3日分の株価から次の日の株価を予測する学習データを作る。

<span style="color:red;">
1～3日目 → 4日目を予測<br>
2～4日目 → 5日目を予測<br>
3～5日目 → 6日目を予測
</span>

→ 20日分のデータから、**17個の学習サンプル**を作成する。

In [ ]:
X, y = [], []

for i in range(len(price) - 3):
    X.append(price[i:i+3])  # 過去3日分
    y.append(price[i+3])    # 次の日の正解値

X = np.array(X)
y = np.array(y)

print('X.shape:', X.shape) 
print('y:', y)

X.shape: (17, 3)
y: [0.7        0.71333333 0.70666667 0.73333333 0.74666667 0.76666667
 0.75333333 0.78666667 0.8        0.79333333 0.82       0.83333333
 0.85333333 0.84       0.86666667 0.88666667 0.9       ]


<br>

#### ④ RNN入力形式に変換

RNNでは、入力データを基本的に**3次元形式**で扱う。

<span style="color:red;">
学習サンプル数 × 時系列の長さ × 特徴量数
</span>

<br> `reshape(-1, 3, 1)`

- `-1`：学習サンプル数を自動計算  
  `1`の場合：サンプル数を1個に固定
- `3`：過去3日分
- `1`：各時点の特徴量（株価）1個

→ `(17, 3)` を **`(17, 3, 1)`** に変換する。

In [5]:
X = X.reshape(-1, 3, 1)

<br>

#### ⑤ RNNモデル構築・学習設定

過去3日分の株価を入力し、**次の日の株価1個を予測する回帰モデル**を構築する。

In [6]:
model = Sequential([
    Input(shape=(3, 1)),  # 3時点 × 特徴量1個
    SimpleRNN(10),        # RNN層：10ニューロン
    Dense(1)              # 次の日の株価を1つ予測
])

model.compile(
    optimizer='adam',
    loss='mse'
)

<br>

#### ⑥ モデル学習

作成した17個の学習サンプルを使って、RNNモデルを学習する。

In [7]:
model.fit(
    X,
    y,
    epochs=100,  # 全学習データを100回繰り返す
    verbose=0
)

<br>

#### ⑦ 次の日の株価を予測

全データの**最後の3日分**を入力し、その次の日の株価を予測する。

<span style="color:red;">
この例には実際の「次の日」の株価がないため、予測値の正確さを評価することはできない。
</span>

In [8]:
last_days = price[-3:].reshape(1, 3, 1)

pred = model.predict(last_days, verbose=0)

# 正規化した値を元の株価単位に戻す
print(f"予測株価：{pred[0][0] * 150:.2f}")

予測株価：134.31


<Br>

#### ※ RNN株価予測の実習フロー

<table align="left" style="
width:950px;
max-width:100%;
border-collapse:collapse;
color:#000;
text-align:left;
margin-left:0 !important;
margin-right:auto !important;
">

<thead>
<tr style="background-color:#dfe3e8; color:#000;">
<th style="padding:8px 16px; width:70px; text-align:center;"><strong>段階</strong></th>
<th style="padding:8px 16px; width:280px; text-align:center;"><strong>作業</strong></th>
<th style="padding:8px 16px; text-align:center;"><strong>内容</strong></th>
</tr>
</thead>

<tbody>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>①</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>株価データ準備</strong></td>
<td style="padding:8px 16px; text-align:left;">RNN回帰の学習に使用する仮想の株価データを用意する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>②</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>正規化</strong></td>
<td style="padding:8px 16px; text-align:left;">株価の値を小さな範囲に変換し、学習を安定させる。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>③</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>学習データ作成</strong></td>
<td style="padding:8px 16px; text-align:left;">スライディングウィンドウで、過去3日 → 次の日の学習サンプルを作成する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>④</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>RNN入力形式に変換</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>reshape(-1, 3, 1)</code>で3次元形式に変換する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑤</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>RNNモデル構築・学習設定</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>SimpleRNN</code>と<code>Dense</code>で回帰モデルを構築し、学習方法を設定する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑥</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>モデル学習</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>fit()</code>で作成した学習データを使ってRNNを学習する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑦</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>次の日の株価予測</strong></td>
<td style="padding:8px 16px; text-align:left;">最後の3日分を入力し、<code>predict()</code>で次の日の株価を予測する。</td>
</tr>

</tbody>
</table>

<br clear="all">